# Shape-Space Velocity Comparison, Part 2

This standalone notebook performs held-out test-subject analysis for old subset `64/128/64`, new subset `64/160/32`, and additive-best. It fits baseline anchors, compares component speed to real finite-difference speed, performs label-free future generation for `64/160/32`, and summarizes decoder Jacobian sensitivity.


In [1]:
# This cell defines the comparison scope and loads all common libraries.
# The notebook compares three best saved models: old subset 64/128/64,
# new baseline-aligned subset 64/160/32, and additive strong best.
import json
import html as html_lib
import math
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import trimesh
from IPython.display import display, Markdown, HTML
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from skimage.measure import marching_cubes

ROOT = Path('/home/jakaria/INR/Deep3DComp')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from deep_sdf import data as deep_sdf_data
from networks.deep_sdf_decoder import Decoder
from networks.longitudinal_disentangled_flow_64_128_64_adv import (
    build_temporal_flow as build_subset128_flow,
)
from networks.longitudinal_flow_64_160_32_pred_dx import (
    build_temporal_flow as build_subset160_flow,
)
from networks.longitudinal_additive_flow import (
    build_temporal_flow as build_additive_flow,
)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPU_ID = 0
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

# Geometry and derivative settings. Increase GRID_RES only after the notebook runs once.
GRID_RES = 64
GRID_BATCH = 2**17
SURFACE_BATCH = 4096
DELTA_T = 0.10
FD_EPS = 1e-3
EVAL_TIME = 0.50
DIAGNOSIS_FOR_COMPONENT_MAPS = 1.0
COMPONENTS = ('age', 'disease_raw', 'disease', 'residual', 'total')

NOTEBOOK_TITLE = 'Shape-Space Velocity Comparison Part 2'
OUTPUT_DIR = ROOT / 'analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Match the previous HTML-report notebooks: save heavy figures externally and keep
# notebook outputs light. Run the final index cell after figure/table cells finish.
SAVE_INTERACTIVE_HTML = True
SHOW_INLINE_PLOTLY = False
SHOW_INLINE_MATPLOTLIB = False
FIGURE_MANIFEST = {}


def figure_slug(value):
    text = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('_')
    return text or 'output'


def register_output(path, title, category, description=''):
    path = Path(path)
    FIGURE_MANIFEST[str(path)] = {
        'path': path,
        'title': str(title),
        'category': str(category),
        'description': str(description),
    }
    print('Saved output:', path)


def html_page(title, body, description=''):
    desc = f"<p class='note'>{html_lib.escape(str(description))}</p>" if description else ''
    return f"""<!doctype html>
<html><head><meta charset='utf-8'><title>{html_lib.escape(str(title))}</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1200px;margin:32px auto;padding:0 22px;color:#202020}}
h1{{margin-bottom:8px}} .note{{background:#edf6f5;border-left:4px solid #147d92;padding:12px 14px;border-radius:6px}}
table{{border-collapse:collapse;width:100%;font-size:13px}} th,td{{border:1px solid #ddd;padding:7px 9px;text-align:left}}
th{{background:#f2eadc}} img{{max-width:100%;border:1px solid #ddd;border-radius:8px}} code{{background:#f3f3f3;padding:2px 5px;border-radius:3px}}
a{{color:#0b5cad;text-decoration:none}} a:hover{{text-decoration:underline}}
</style></head><body><h1>{html_lib.escape(str(title))}</h1>{desc}{body}</body></html>"""


def save_table(df, filename, title, category, description=''):
    stem = figure_slug(filename)
    html_path = FIGURE_DIR / f'{stem}.html'
    csv_path = FIGURE_DIR / f'{stem}.csv'
    df.to_csv(csv_path, index=False)
    body = (
        f"<p><strong>CSV:</strong> <a href='{csv_path.name}'>{csv_path.name}</a></p>"
        + df.to_html(index=False, escape=False)
    )
    html_path.write_text(html_page(title, body, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    return html_path


def save_plotly_figure(fig, filename, title, category, description='', show_inline=False):
    html_path = FIGURE_DIR / f'{figure_slug(filename)}.html'
    if SAVE_INTERACTIVE_HTML:
        fig.write_html(html_path, include_plotlyjs='directory', full_html=True, auto_open=False)
        register_output(html_path, title, category, description)
    if SHOW_INLINE_PLOTLY and show_inline:
        fig.show()
    return html_path


def save_matplotlib_figure(fig, filename, title, category, description='', dpi=170):
    stem = figure_slug(filename)
    png_path = FIGURE_DIR / f'{stem}.png'
    html_path = FIGURE_DIR / f'{stem}.html'
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    body = f"<p><img src='{png_path.name}' alt='{html_lib.escape(str(title))}'></p>"
    html_path.write_text(html_page(title, body, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    if SHOW_INLINE_MATPLOTLIB:
        plt.show()
    else:
        plt.close(fig)
    return html_path


def write_figure_index(report_title=NOTEBOOK_TITLE):
    entries = sorted(
        FIGURE_MANIFEST.values(),
        key=lambda item: (item['category'], item['title'], str(item['path'])),
    )
    groups = {}
    for item in entries:
        groups.setdefault(item['category'], []).append(item)
    chunks = [
        "<!doctype html><html><head><meta charset='utf-8'>",
        f"<title>{html_lib.escape(report_title)}</title>",
        "<style>body{font-family:Arial,sans-serif;max-width:1150px;margin:32px auto;padding:0 20px;color:#202020}",
        "h1{margin-bottom:8px}h2{margin-top:30px}li{margin:10px 0}a{color:#0b5cad;text-decoration:none}",
        "a:hover{text-decoration:underline}.desc{color:#555;font-size:14px}</style></head><body>",
        f"<h1>{html_lib.escape(report_title)}</h1>",
        "<p>All heavy notebook outputs are saved here as HTML pages. Open Plotly pages in a browser for interactive 3D inspection.</p>",
        f"<p><strong>Output directory:</strong> <code>{html_lib.escape(str(FIGURE_DIR))}</code></p>",
    ]
    for category, items in groups.items():
        chunks.append(f"<h2>{html_lib.escape(category)}</h2><ul>")
        for item in items:
            relative = item['path'].relative_to(FIGURE_DIR)
            desc = html_lib.escape(item.get('description', ''))
            chunks.append(
                f"<li><a href='{relative.as_posix()}'>{html_lib.escape(item['title'])}</a>"
                f"<br><span class='desc'>{desc}</span></li>"
            )
        chunks.append('</ul>')
    chunks.append('</body></html>')
    index_path = FIGURE_DIR / 'index.html'
    index_path.write_text(''.join(chunks), encoding='utf-8')
    print('Figure index:', index_path)
    return index_path


BASE = ROOT / 'examples' / 'Torus_subset_100_id_age_progression'
SUBSET128_DIR = BASE / 'longitudinal_age_disease_conditioned_cocycle_shape_pair_loss_multiple_pairs_disentangled_velocity_64_128_64_binary_margin_adv'
SUBSET160_DIR = BASE / 'velocity_64_160_32_baseline_dx'
ADDITIVE_DIR = BASE / 'longitudinal_age_disease_additive_velocity_disentanglement_strong_residual_invariance_adv'

# Best saved checkpoints based on the results currently present:
# - 64/128/64: epoch 500 is the best old GT-forecast checkpoint.
# - 64/160/32: epoch 1000 is the best saved E2 forecast checkpoint; epoch 1500 had better disease CSV but no saved model checkpoint.
# - additive: epoch 1000 is the additive strong best checkpoint used in prior notebooks.
MODEL_CONFIGS = {
    'subset64_128_best': {
        'label': 'Subset 64/128/64 best',
        'short_label': '64/128/64',
        'kind': 'subset128',
        'exp_dir': SUBSET128_DIR,
        'checkpoint': '500',
        'reported_test_chamfer': 0.00014264255878515542,
        'selection_note': 'Best old subset checkpoint from GT ranking.',
    },
    'subset64_160_best_saved': {
        'label': 'Subset 64/160/32 E2 best saved',
        'short_label': '64/160/32',
        'kind': 'subset160',
        'exp_dir': SUBSET160_DIR,
        'checkpoint': '1000',
        'reported_test_chamfer': 0.00018364414945436707,
        'selection_note': 'Best saved baseline-aligned forecast checkpoint; epoch 1500 diagnosis CSV exists but no checkpoint file exists.',
    },
    'additive_best': {
        'label': 'Additive strong best',
        'short_label': 'additive',
        'kind': 'additive',
        'exp_dir': ADDITIVE_DIR,
        'checkpoint': '1000',
        'reported_test_chamfer': 0.00021775640198029578,
        'selection_note': 'Best additive strong checkpoint from prior comparison.',
    },
}

print('Device:', DEVICE)
print('Grid resolution:', GRID_RES)
print('Finite-step interval DELTA_T:', DELTA_T)
print('Evaluation time:', EVAL_TIME, '| forced disease condition for component maps:', DIAGNOSIS_FOR_COMPONENT_MAPS)

from train_deep_sdf_longitudinal_disentangled_flow_64_128_64_adv import (
    optimize_subject_anchor_from_observations as optimize_subset128_anchor,
)
from train_deep_sdf_longitudinal_flow_64_160_32_baseline_dx import (
    optimize_subject_anchor_from_observations as optimize_subset160_anchor,
)
from train_deep_sdf_longitudinal_additive_flow import (
    optimize_subject_anchor_from_observations as optimize_additive_anchor,
)

RUN_HELDOUT_ANCHORS = True
RUN_64_160_FUTURE_MESH_VIS = True
TEST_ANCHOR_STEPS = 300
TEST_ANCHOR_SAMPLES = 4096
FUTURE_SDF_SAMPLES = 8192


Device: cuda:0
Grid resolution: 64
Finite-step interval DELTA_T: 0.1
Evaluation time: 0.5 | forced disease condition for component maps: 1.0


## Load Same Best Saved Checkpoints

This repeats Part 1 loading so Part 2 can run independently.

In [2]:
# This cell loads checkpoint-specific decoders, temporal flows, and mean training anchors.
# The mean anchor is used only for population-level shape-space velocity maps.
def checkpoint_path(exp_dir, subdir, checkpoint):
    name = str(checkpoint)
    if not name.endswith('.pth'):
        name += '.pth'
    path = Path(exp_dir) / subdir / name
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def strip_module_prefix(state):
    return {k.removeprefix('module.'): v for k, v in state.items()}


def latent_weights(payload):
    lat = payload.get('latent_codes_state_dict', payload.get('latent_codes'))
    if isinstance(lat, dict):
        lat = lat['weight']
    if lat.ndim == 3 and lat.shape[1] == 1:
        lat = lat[:, 0]
    return lat.detach().float()


def flow_builder(kind):
    if kind == 'subset128':
        return build_subset128_flow
    if kind == 'subset160':
        return build_subset160_flow
    if kind == 'additive':
        return build_additive_flow
    raise ValueError(f'Unknown model kind: {kind}')


def load_model(config):
    exp_dir = Path(config['exp_dir'])
    specs = json.loads((exp_dir / 'specs.json').read_text())
    decoder = Decoder(int(specs['CodeLength']), **specs['NetworkSpecs']).to(DEVICE)
    flow = flow_builder(config['kind'])(
        specs,
        int(specs['CodeLength']),
        list(specs.get('FlowHiddenDims', [256, 256])),
        age_condition_dim=int(specs.get('AgeConditionDim', 0)),
    ).to(DEVICE)

    model_payload = torch.load(
        checkpoint_path(exp_dir, 'ModelParameters', config['checkpoint']),
        map_location=DEVICE,
    )
    decoder.load_state_dict(strip_module_prefix(model_payload['model_state_dict']))
    flow.load_state_dict(strip_module_prefix(model_payload['flow_state_dict']))
    decoder.eval()
    flow.eval()

    latent_payload = torch.load(
        checkpoint_path(exp_dir, 'LatentCodes', config['checkpoint']),
        map_location='cpu',
    )
    weights = latent_weights(latent_payload)
    mean_z = weights.mean(dim=0, keepdim=True).to(DEVICE)

    return {
        **config,
        'specs': specs,
        'decoder': decoder,
        'flow': flow,
        'mean_z': mean_z,
        'latent_weights': weights,
        'epoch': int(model_payload.get('epoch', -1)),
    }


MODELS = {name: load_model(config) for name, config in MODEL_CONFIGS.items()}

checkpoint_table = pd.DataFrame([
    {
        'model_key': name,
        'model': model['label'],
        'checkpoint': model['checkpoint'],
        'loaded_epoch': model['epoch'],
        'velocity_blocks': (
            model['specs'].get('VelocityAgeDim'),
            model['specs'].get('VelocityDiseaseDim'),
            model['specs'].get('VelocityResidualDim'),
        ),
        'reported_or_current_test_chamfer': model['reported_test_chamfer'],
        'mean_z_norm': float(model['mean_z'].norm().item()),
        'selection_note': model['selection_note'],
    }
    for name, model in MODELS.items()
])
save_table(
    checkpoint_table,
    '01_checkpoint_selection',
    'Checkpoint selection',
    'Model setup',
    'Loaded checkpoints, velocity block dimensions, mean-anchor norms, and selection notes.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/01_checkpoint_selection.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/01_checkpoint_selection.html')

## Shared Geometry/Jacobian Utilities

Same component adapter as Part 1: subset models are zero-padded into the 256D latent space; additive components are already full 256D vectors.

In [3]:
# This cell defines decoder-Jacobian shape-speed utilities.
# It converts a latent velocity vector v into normal surface speed using the INR Jacobian.
def decode_points(decoder, z, xyz, batch_size=GRID_BATCH):
    z = z.reshape(1, -1)
    outputs = []
    with torch.no_grad():
        for start in range(0, xyz.shape[0], batch_size):
            x = xyz[start:start + batch_size].to(DEVICE)
            zz = z.expand(x.shape[0], -1)
            outputs.append(decoder(torch.cat([zz, x], dim=1)).squeeze(-1).cpu())
    return torch.cat(outputs).numpy()


def regular_grid(resolution=GRID_RES):
    axis = torch.linspace(-1.0, 1.0, int(resolution), dtype=torch.float32)
    grid = torch.stack(torch.meshgrid(axis, axis, axis, indexing='ij'), dim=-1).reshape(-1, 3)
    return axis.numpy(), grid


def decode_grid(decoder, z, resolution=GRID_RES):
    axis, xyz = regular_grid(resolution)
    values = decode_points(decoder, z, xyz).reshape(resolution, resolution, resolution)
    return axis, values


def mesh_from_grid(sdf_grid):
    spacing = 2.0 / (sdf_grid.shape[0] - 1)
    vertices, faces, _, _ = marching_cubes(sdf_grid, level=0.0, spacing=(spacing, spacing, spacing))
    vertices = vertices - 1.0
    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    if mesh.is_watertight and mesh.volume < 0.0:
        mesh.invert()
    return mesh


def _zero_pad_subset(model, comp, block):
    flow = model['flow']
    batch = comp.shape[0]
    z = comp.new_zeros(batch, flow.age_dim + flow.disease_dim + flow.residual_dim)
    if block == 'age':
        z[:, :flow.age_dim] = comp
    elif block == 'disease':
        z[:, flow.age_dim:flow.age_dim + flow.disease_dim] = comp
    elif block == 'residual':
        z[:, flow.age_dim + flow.disease_dim:] = comp
    else:
        raise ValueError(block)
    return z


def component_vectors(model, z, time_value=EVAL_TIME, diagnosis=DIAGNOSIS_FOR_COMPONENT_MAPS):
    flow = model['flow']
    t = torch.full((z.shape[0], 1), float(time_value), device=z.device, dtype=z.dtype)
    cond = torch.full((z.shape[0], 1), float(diagnosis), device=z.device, dtype=z.dtype)
    with torch.no_grad():
        comp = flow.velocity_components(z, t, t, age_cond=cond)

    if model['kind'] in ('subset128', 'subset160'):
        age = _zero_pad_subset(model, comp['age'], 'age')
        disease_raw = _zero_pad_subset(model, comp.get('disease_raw', comp['disease']), 'disease')
        disease = _zero_pad_subset(model, comp['disease'], 'disease')
        residual = _zero_pad_subset(model, comp['residual'], 'residual')
    else:
        age = comp['age']
        disease = comp['disease']
        disease_raw = comp.get('disease_raw', disease)
        residual = comp['residual']
    return {
        'age': age,
        'disease_raw': disease_raw,
        'disease': disease,
        'residual': residual,
        'total': age + disease + residual,
    }


def surface_geometry(decoder, z, resolution=GRID_RES):
    axis, grid = decode_grid(decoder, z, resolution)
    mesh = mesh_from_grid(grid)
    vertices = torch.as_tensor(mesh.vertices, dtype=z.dtype, device=DEVICE)

    sdf_parts = []
    grad_parts = []
    for start in range(0, vertices.shape[0], SURFACE_BATCH):
        x = vertices[start:start + SURFACE_BATCH].detach().clone().requires_grad_(True)
        zz = z.detach().expand(x.shape[0], -1)
        sdf = decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        grad_x = torch.autograd.grad(sdf.sum(), x, create_graph=False)[0]
        sdf_parts.append(sdf.detach().cpu())
        grad_parts.append(grad_x.detach().cpu())

    sdf = torch.cat(sdf_parts).numpy()
    grad_x = torch.cat(grad_parts).numpy()
    grad_norm = np.linalg.norm(grad_x, axis=1) + 1e-12
    grad_unit = grad_x / grad_norm[:, None]
    mesh_normals = np.asarray(mesh.vertex_normals)
    alignment = np.einsum('ij,ij->i', grad_unit, mesh_normals)
    sdf_to_outward_sign = 1.0 if np.nanmedian(alignment) >= 0.0 else -1.0
    outward_normals = sdf_to_outward_sign * grad_unit

    return {
        'axis': axis,
        'sdf_grid': grid,
        'mesh': mesh,
        'vertices': vertices,
        'base_sdf': sdf,
        'grad_x': grad_x,
        'grad_norm': grad_norm,
        'outward_normals': outward_normals,
        'sdf_to_outward_sign': sdf_to_outward_sign,
        'median_normal_alignment': float(np.nanmedian(np.abs(alignment))),
        'signed_mesh_volume': float(mesh.volume),
        'mesh_is_watertight': bool(mesh.is_watertight),
    }


def latent_jvp_at_vertices(decoder, z, velocity, vertices):
    parts = []
    for start in range(0, vertices.shape[0], SURFACE_BATCH):
        x = vertices[start:start + SURFACE_BATCH].detach()
        z0 = z.detach().clone().requires_grad_(True)
        v0 = velocity.detach()
        def decode_for_latent(latent):
            zz = latent.expand(x.shape[0], -1)
            return decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        _, jvp = torch.autograd.functional.jvp(decode_for_latent, z0, v0, create_graph=False, strict=False)
        parts.append(jvp.detach().cpu())
    return torch.cat(parts).numpy()


def vertex_area_weights(mesh):
    weights = np.zeros(len(mesh.vertices), dtype=np.float64)
    contribution = np.asarray(mesh.area_faces, dtype=np.float64) / 3.0
    for corner in range(3):
        np.add.at(weights, np.asarray(mesh.faces)[:, corner], contribution)
    return weights


def surface_component_map(model, geometry, z, velocity, delta_t=DELTA_T, fd_eps=FD_EPS):
    vertices = geometry['vertices']
    decoder = model['decoder']
    jvp = latent_jvp_at_vertices(decoder, z, velocity, vertices)

    xyz_cpu = vertices.detach().cpu()
    sdf_step = decode_points(decoder, z + delta_t * velocity, xyz_cpu)
    sdf_plus = decode_points(decoder, z + fd_eps * velocity, xyz_cpu)
    sdf_minus = decode_points(decoder, z - fd_eps * velocity, xyz_cpu)
    fd_directional = (sdf_plus - sdf_minus) / (2.0 * fd_eps)

    sign = geometry['sdf_to_outward_sign']
    grad_norm = geometry['grad_norm']
    normal_speed = -sign * jvp / grad_norm
    normal_speed_fd = -sign * fd_directional / grad_norm
    inr_difference = geometry['base_sdf'] - sdf_step
    arrows = normal_speed[:, None] * geometry['outward_normals']

    weights = vertex_area_weights(geometry['mesh'])
    total_area = weights.sum()
    jvp_error = np.sqrt(np.average((jvp - fd_directional) ** 2, weights=weights))
    jvp_scale = np.sqrt(np.average(fd_directional ** 2, weights=weights)) + 1e-12
    metrics = {
        'latent_velocity_norm': float(velocity.norm().item()),
        'mean_abs_normal_speed': float(np.average(np.abs(normal_speed), weights=weights)),
        'rms_normal_speed': float(np.sqrt(np.average(normal_speed ** 2, weights=weights))),
        'net_volume_rate': float(np.sum(normal_speed * weights)),
        'expanding_area_fraction': float(np.sum(weights[normal_speed > 0]) / total_area),
        'contracting_area_fraction': float(np.sum(weights[normal_speed < 0]) / total_area),
        'jvp_fd_relative_rmse': float(jvp_error / jvp_scale),
    }
    return {
        'velocity': velocity.detach(),
        'jvp': jvp,
        'fd_directional': fd_directional,
        'normal_speed': normal_speed,
        'normal_speed_fd': normal_speed_fd,
        'inr_difference': inr_difference,
        'sdf_step': sdf_step,
        'arrows': arrows,
        'metrics': metrics,
    }


def weighted_correlation(a, b, weights):
    w = weights / weights.sum()
    am = np.sum(w * a)
    bm = np.sum(w * b)
    ac = a - am
    bc = b - bm
    denom = math.sqrt(np.sum(w * ac**2) * np.sum(w * bc**2)) + 1e-12
    return float(np.sum(w * ac * bc) / denom)


## Test Metadata and Anchor Fitting

The `64/160/32` model uses predicted-soft diagnosis from the optimized baseline anchor. The other two models require oracle diagnosis because they do not have an anchor disease head.

In [4]:
# This cell loads test metadata and defines anchor fitting for each model.
# Old subset/additive use oracle diagnosis because they do not have a test-time disease head.
# New 64/160/32 uses predicted_soft disease condition from its optimized baseline anchor.
def load_labels(labels_path):
    obj = torch.load(labels_path, map_location='cpu')
    rows = []
    for scan, payload in obj.items():
        row = {'scan_id': str(scan)}
        row.update(payload)
        rows.append(row)
    return pd.DataFrame(rows)


def parse_scan(scan):
    match = re.match(r'^(?:ID|id)_(\d+)_t(\d+)$', Path(str(scan)).stem)
    if match is None:
        raise ValueError(scan)
    return int(match.group(1)), int(match.group(2))


def test_records(specs):
    labels = load_labels(specs['AgeMetadataFile'])
    label_map = labels.set_index('scan_id').to_dict('index')
    split = json.loads(Path(specs['TestSplit']).read_text())
    records = []
    for scan in split:
        stem = Path(str(scan)).stem
        sid, tp = parse_scan(stem)
        meta = label_map[stem]
        records.append({
            'scan_id': stem,
            'sid': sid,
            'tp': tp,
            'time': float(meta['age_norm']),
            'age': float(meta['age']),
            'diagnosis': int(meta['diagnosis']),
            'thickness': float(meta['thickness']),
            'bump_height': float(meta['bump_height']),
            'mesh_path': Path(specs['AgeMetadataFile']).parent / str(meta['mesh_path']),
            'sdf_path': Path(specs['DataSource']) / f'{stem}.npz',
        })
    by_subject = {}
    for record in records:
        by_subject.setdefault(record['sid'], []).append(record)
    return {sid: sorted(items, key=lambda x: x['time']) for sid, items in by_subject.items()}


def choose_test_subjects(records_by_subject):
    healthy = next(sid for sid, recs in records_by_subject.items() if int(recs[0]['diagnosis']) == 0)
    diseased = next(sid for sid, recs in records_by_subject.items() if int(recs[0]['diagnosis']) == 1)
    return healthy, diseased


def read_sdf_samples(record):
    return deep_sdf_data.read_sdf_samples_into_ram(str(record['sdf_path']))


def infer_test_anchor(model, observation_records):
    observations = []
    for record in observation_records:
        observations.append({
            'samples': read_sdf_samples(record),
            'time': float(record['time']),
            'age_cond': torch.tensor([float(record['diagnosis'])], dtype=torch.float32),
        })
    common = dict(
        decoder=model['decoder'],
        temporal_flow=model['flow'],
        latent_size=int(model['specs']['CodeLength']),
        observations=observations,
        clamp_dist=float(model['specs'].get('ClampingDistance', 0.1)),
        num_iterations=TEST_ANCHOR_STEPS,
        num_samples=TEST_ANCHOR_SAMPLES,
        lr=5e-3,
        init_std=0.01,
        code_reg_lambda=float(model['specs'].get('CodeRegularizationLambda', 1e-4)),
        code_bound=model['specs'].get('CodeBound', None),
        use_pair_forward_consistency=False,
        use_pair_backward_consistency=False,
        use_general_cocycle_consistency=False,
        use_age_conditioning=True,
    )
    if model['kind'] == 'subset160':
        anchor, history = optimize_subset160_anchor(
            **common,
            disease_condition_mode='predicted_soft',
            disease_threshold=float(model['specs'].get('AnchorDiseaseThreshold', 0.5)),
        )
    elif model['kind'] == 'subset128':
        anchor, history = optimize_subset128_anchor(**common)
    else:
        anchor, history = optimize_additive_anchor(**common)
    return anchor.detach(), history


def inferred_gate(model, anchor, true_diagnosis=None):
    if model['kind'] == 'subset160' and callable(getattr(model['flow'], 'anchor_disease_probability', None)):
        with torch.no_grad():
            prob = float(model['flow'].anchor_disease_probability(anchor).view(-1)[0].item())
        return prob, 'predicted_soft'
    return float(true_diagnosis), 'oracle_for_model_without_anchor_head'

RECORDS = test_records(MODELS['subset64_160_best_saved']['specs'])
healthy_sid, diseased_sid = choose_test_subjects(RECORDS)
print('Selected healthy SID:', healthy_sid, '| diseased SID:', diseased_sid)


Selected healthy SID: 27 | diseased SID: 4


## Held-Out Component Maps

This computes component velocity maps after fitting each model to only the first held-out scan.

In [5]:
# This cell fits held-out baseline anchors and computes component maps at each subject baseline.
# For 64/160/32, the condition is inferred from the anchor; for the other two models, condition is oracle.
if RUN_HELDOUT_ANCHORS:
    HELDOUT = {}
    heldout_rows = []
    for sid in (healthy_sid, diseased_sid):
        baseline = RECORDS[sid][0]
        HELDOUT[sid] = {}
        for model_name, model in MODELS.items():
            print('Fitting', model['label'], '| SID', sid)
            anchor, history = infer_test_anchor(model, [baseline])
            gate, gate_mode = inferred_gate(model, anchor, baseline['diagnosis'])
            geometry = surface_geometry(model['decoder'], anchor)
            velocities = component_vectors(model, anchor, time_value=baseline['time'], diagnosis=gate)
            maps = {component: surface_component_map(model, geometry, anchor, velocities[component]) for component in COMPONENTS}
            HELDOUT[sid][model_name] = {
                'baseline': baseline,
                'anchor': anchor,
                'loss_history': history,
                'gate': gate,
                'gate_mode': gate_mode,
                'geometry': geometry,
                'maps': maps,
            }
            for component in COMPONENTS:
                heldout_rows.append({
                    'sid': sid,
                    'true_diagnosis': baseline['diagnosis'],
                    'model_key': model_name,
                    'model': model['label'],
                    'gate_used': gate,
                    'gate_mode': gate_mode,
                    'component': component,
                    **maps[component]['metrics'],
                })
    HELDOUT_SPEED = pd.DataFrame(heldout_rows)
    save_table(
        HELDOUT_SPEED,
        '02_heldout_component_speed_table',
        'Held-out component speed table',
        'Held-out speed',
        'One baseline scan is fitted per selected test subject; 64/160/32 uses inferred disease probability.',
    )
else:
    print('Skipped held-out anchor fitting. Set RUN_HELDOUT_ANCHORS=True.')


Fitting Subset 64/128/64 best | SID 27
Fitting Subset 64/160/32 E2 best saved | SID 27
Fitting Additive strong best | SID 27
Fitting Subset 64/128/64 best | SID 4
Fitting Subset 64/160/32 E2 best saved | SID 4
Fitting Additive strong best | SID 4
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/02_heldout_component_speed_table.html


## Visual Held-Out Disentanglement

Rows are models, columns are components. For `64/160/32`, disease is gated by the inferred disease probability.

In [6]:
# This cell plots held-out subject component maps.
# It shows whether the model puts progression into age, disease, or residual after baseline-only test fitting.
def robust_symmetric_limit(arrays, quantile=0.98):
    vals = np.concatenate([np.ravel(a[np.isfinite(a)]) for a in arrays if np.size(a)])
    if vals.size == 0:
        return 1.0
    limit = float(np.quantile(np.abs(vals), quantile))
    return limit if limit > 0 else 1.0


def mesh_surface_trace(mesh, values, cmin, cmax, colorscale='RdBu_r', show_colorbar=False, title=''):
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        intensity=np.asarray(values),
        colorscale=colorscale,
        cmin=cmin, cmax=cmax,
        showscale=show_colorbar,
        colorbar=dict(title=title, thickness=18, len=0.72),
        opacity=1.0,
        flatshading=False,
        lighting=dict(ambient=0.58, diffuse=0.72, specular=0.18, roughness=0.55),
    )


def heldout_surface_figure(sid, value_key='normal_speed', components=('age', 'disease', 'residual', 'total')):
    arrays = [HELDOUT[sid][name]['maps'][component][value_key] for name in MODELS for component in components]
    limit = robust_symmetric_limit(arrays)
    baseline = RECORDS[sid][0]
    fig = make_subplots(
        rows=len(MODELS), cols=len(components),
        specs=[[{'type': 'scene'}] * len(components) for _ in MODELS],
        row_titles=[MODELS[name]['label'] for name in MODELS],
        column_titles=list(components),
        horizontal_spacing=0.01, vertical_spacing=0.02,
    )
    for row, name in enumerate(MODELS, start=1):
        geometry = HELDOUT[sid][name]['geometry']
        maps = HELDOUT[sid][name]['maps']
        for col, component in enumerate(components, start=1):
            fig.add_trace(
                mesh_surface_trace(
                    geometry['mesh'], maps[component][value_key], -limit, limit,
                    show_colorbar=(row == 1 and col == len(components)),
                    title=value_key.replace('_', ' '),
                ),
                row=row, col=col,
            )
    for scene_name in [k for k in fig.layout if str(k).startswith('scene')]:
        fig.layout[scene_name].update(
            aspectmode='data',
            xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
            camera=dict(eye=dict(x=1.55, y=1.55, z=1.05)),
        )
    status = 'diseased' if baseline['diagnosis'] else 'healthy'
    fig.update_layout(
        title=(
            f'Held-out SID {sid} ({status}), baseline age {baseline["age"]:.2f}'
            f'<br><sup>{value_key}; 64/160/32 uses inferred disease probability when shown.</sup>'
        ),
        width=350 * len(components), height=330 * len(MODELS),
        margin=dict(l=130, r=30, t=90, b=20),
    )
    return fig

for sid in (healthy_sid, diseased_sid):
    save_plotly_figure(
        heldout_surface_figure(sid, 'normal_speed'),
        f'03_heldout_sid_{sid}_normal_speed',
        f'Held-out SID {sid} normal speed maps',
        'Held-out surface maps',
        'Subject-level decoder-Jacobian normal speed after baseline-only test anchor fitting.',
    )
    save_plotly_figure(
        heldout_surface_figure(sid, 'inr_difference'),
        f'04_heldout_sid_{sid}_inr_difference',
        f'Held-out SID {sid} INR difference maps',
        'Held-out surface maps',
        'Finite-step INR difference after baseline-only test anchor fitting.',
    )


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/03_heldout_sid_27_normal_speed.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/04_heldout_sid_27_inr_difference.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/03_heldout_sid_4_normal_speed.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/04_heldout_sid_4_inr_difference.html


## Compare Model Speeds With Real Subject Speeds

Real speed is finite-difference thickness/bump change from the actual scans. Model speed is the decoder-induced normal speed at the baseline surface.

In [7]:
# This cell compares held-out model speeds against real finite-difference speeds for the same subjects.
# Real speed uses observed thickness and bump changes from consecutive scans; model speed uses baseline component maps.
def real_subject_rates(records):
    rows = []
    for left, right in zip(records[:-1], records[1:]):
        dt_norm = float(right['time']) - float(left['time'])
        dt_year = float(right['age']) - float(left['age'])
        if abs(dt_norm) < 1e-8:
            continue
        rows.append({
            'sid': left['sid'],
            'diagnosis': left['diagnosis'],
            'from_tp': left['tp'],
            'to_tp': right['tp'],
            'age_mid': 0.5 * (left['age'] + right['age']),
            'dt_years': dt_year,
            'dt_norm': dt_norm,
            'real_thickness_rate_norm': (right['thickness'] - left['thickness']) / dt_norm,
            'real_bump_rate_norm': (right['bump_height'] - left['bump_height']) / dt_norm,
        })
    return rows

REAL_HELDOUT_SPEED = pd.DataFrame([row for sid in (healthy_sid, diseased_sid) for row in real_subject_rates(RECORDS[sid])])
save_table(
    REAL_HELDOUT_SPEED,
    '05_heldout_real_speed_table',
    'Held-out real speed table',
    'Real-vs-model speed',
    'Observed finite-difference thickness and bump rates for selected held-out subjects.',
)

model_summary = (
    HELDOUT_SPEED[HELDOUT_SPEED['component'].isin(['age', 'disease', 'residual', 'total'])]
    .pivot_table(index=['sid', 'true_diagnosis', 'model', 'gate_used', 'gate_mode'], columns='component', values='rms_normal_speed')
    .reset_index()
)
model_summary['residual_fraction'] = model_summary['residual'] / (model_summary['total'] + 1e-12)
save_table(
    model_summary,
    '06_heldout_model_speed_summary',
    'Held-out model speed summary',
    'Real-vs-model speed',
    'Component RMS speeds and residual leakage for selected held-out subjects.',
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for sid, sub in REAL_HELDOUT_SPEED.groupby('sid'):
    axes[0].plot(sub['age_mid'], sub['real_thickness_rate_norm'], marker='o', label=f'SID {sid}')
    axes[1].plot(sub['age_mid'], sub['real_bump_rate_norm'], marker='o', label=f'SID {sid}')
for _, row in model_summary.iterrows():
    label = f"{row['model']} | SID {int(row['sid'])}"
    axes[2].scatter(row['disease'], row['residual_fraction'], label=label, s=70)
axes[0].axhline(0, color='black', linewidth=1)
axes[1].axhline(0, color='black', linewidth=1)
axes[0].set_title('Real thickness speed by subject')
axes[1].set_title('Real bump speed by subject')
axes[2].set_title('Model disease speed vs residual leakage')
axes[0].set_ylabel('rate per normalized time')
axes[1].set_ylabel('rate per normalized time')
axes[2].set_xlabel('disease RMS normal speed')
axes[2].set_ylabel('residual / total RMS normal speed')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7)
fig.suptitle('Real speed tells the expected progression direction; model speed tells where each model assigns that progression.')
fig.tight_layout()
save_matplotlib_figure(
    fig,
    '07_real_vs_model_speed',
    'Real speed versus model component speed',
    'Real-vs-model speed',
    'Compares observed subject progression with model disease speed and residual leakage.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/05_heldout_real_speed_table.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/06_heldout_model_speed_summary.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/07_real_vs_model_speed.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/07_real_vs_model_speed.html')

## Label-Free Future Generation for `64/160/32`

This is the key test-time workflow: fit one baseline anchor, infer disease probability, then predict future scans.

In [8]:
# This cell performs label-free future generation for the 64/160/32 model.
# The disease probability is inferred from the baseline-fitted anchor; true diagnosis is used only for reporting after inference.
def transport(model, z0, start_time, target_time, gate):
    s = torch.full((z0.shape[0], 1), float(start_time), device=z0.device, dtype=z0.dtype)
    t = torch.full((z0.shape[0], 1), float(target_time), device=z0.device, dtype=z0.dtype)
    cond = torch.full((z0.shape[0], 1), float(gate), device=z0.device, dtype=z0.dtype)
    with torch.no_grad():
        return z0 + (t - s) * model['flow'](z0, s, t, age_cond=cond)


def held_out_sdf_mae(decoder, latent, record, num_samples=FUTURE_SDF_SAMPLES):
    sdf_data = deep_sdf_data.unpack_sdf_samples_from_ram(read_sdf_samples(record), int(num_samples)).to(DEVICE)
    xyz = sdf_data[:, 0:3]
    sdf_gt = torch.clamp(sdf_data[:, 3].unsqueeze(1), -0.1, 0.1)
    with torch.no_grad():
        pred = decoder(torch.cat([latent.expand(xyz.shape[0], -1), xyz], dim=1))
        pred = torch.clamp(pred, -0.1, 0.1)
    return float(torch.mean(torch.abs(pred - sdf_gt)).item())


def mesh_from_latent(model, latent, grid_res=GRID_RES):
    _, sdf_grid = decode_grid(model['decoder'], latent, grid_res)
    return mesh_from_grid(sdf_grid)

subset160_model = MODELS['subset64_160_best_saved']
future_rows = []
FUTURE_MESHES = {}
for sid in (healthy_sid, diseased_sid):
    recs = RECORDS[sid]
    baseline = recs[0]
    anchor = HELDOUT[sid]['subset64_160_best_saved']['anchor']
    gate = HELDOUT[sid]['subset64_160_best_saved']['gate']
    FUTURE_MESHES[sid] = {'gt': [], 'pred': []}
    for record in recs:
        z_t = transport(subset160_model, anchor, baseline['time'], record['time'], gate)
        mae = held_out_sdf_mae(subset160_model['decoder'], z_t, record)
        row = {
            'sid': sid,
            'true_diagnosis': baseline['diagnosis'],
            'inferred_disease_probability': gate,
            'predicted_disease_class': int(gate >= 0.5),
            'scan_id': record['scan_id'],
            'tp': record['tp'],
            'age': record['age'],
            'phase': 'observed' if record['tp'] == baseline['tp'] else 'forecast',
            'sdf_mae': mae,
        }
        try:
            pred_mesh = mesh_from_latent(subset160_model, z_t)
            row['predicted_volume'] = float(pred_mesh.volume)
            row['gt_volume'] = float(trimesh.load(record['mesh_path'], process=False).volume)
            row['relative_volume_error'] = abs(row['predicted_volume'] - row['gt_volume']) / (abs(row['gt_volume']) + 1e-12)
            FUTURE_MESHES[sid]['pred'].append(pred_mesh)
            FUTURE_MESHES[sid]['gt'].append(trimesh.load(record['mesh_path'], process=False))
        except Exception as exc:
            row['mesh_error'] = str(exc)
        future_rows.append(row)

FUTURE_64_160 = pd.DataFrame(future_rows)
save_table(
    FUTURE_64_160,
    '08_64_160_future_prediction_table',
    '64/160/32 label-free future prediction table',
    'Future generation',
    'Forecast uses one baseline anchor and inferred disease probability; true diagnosis is reported only after inference.',
)

FUTURE_64_160_SUMMARY = (
    FUTURE_64_160.groupby(['sid', 'true_diagnosis', 'inferred_disease_probability', 'predicted_disease_class', 'phase'])
    .agg(mean_sdf_mae=('sdf_mae', 'mean'), mean_relative_volume_error=('relative_volume_error', 'mean'))
    .reset_index()
)
save_table(
    FUTURE_64_160_SUMMARY,
    '09_64_160_future_prediction_summary',
    '64/160/32 future prediction summary',
    'Future generation',
    'SDF MAE and volume error grouped by subject and observed/forecast phase.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/08_64_160_future_prediction_table.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/09_64_160_future_prediction_summary.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/09_64_160_future_prediction_summary.html')

## GT vs Predicted Future Meshes

The prediction row uses inferred disease probability, not the test label.

In [9]:
# This cell visualizes label-free 64/160/32 future generation as GT-vs-predicted meshes.
# It uses the inferred disease probability from the baseline anchor, not the true disease label.
def plain_mesh_trace(mesh, color, name, opacity=1.0, showlegend=False):
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color, opacity=opacity, name=name, showlegend=showlegend,
        lighting=dict(ambient=0.62, diffuse=0.72, specular=0.15, roughness=0.6),
    )


def future_mesh_figure(sid):
    recs = RECORDS[sid]
    meshes = FUTURE_MESHES[sid]
    n = min(len(recs), len(meshes['gt']), len(meshes['pred']))
    if n == 0:
        raise RuntimeError(f'No meshes available for SID {sid}.')
    fig = make_subplots(
        rows=2, cols=n,
        specs=[[{'type': 'scene'}] * n, [{'type': 'scene'}] * n],
        row_titles=['GT real scan', 'Predicted label-free'],
        column_titles=[f"t{recs[i]['tp']} age {recs[i]['age']:.1f}" for i in range(n)],
        horizontal_spacing=0.01, vertical_spacing=0.02,
    )
    for col in range(n):
        fig.add_trace(plain_mesh_trace(meshes['gt'][col], '#A7C7E7', 'GT'), row=1, col=col+1)
        fig.add_trace(plain_mesh_trace(meshes['pred'][col], '#D1495B', 'Pred'), row=2, col=col+1)
    for scene_name in [k for k in fig.layout if str(k).startswith('scene')]:
        fig.layout[scene_name].update(
            aspectmode='data',
            xaxis_visible=False, yaxis_visible=False, zaxis_visible=False,
            camera=dict(eye=dict(x=1.55, y=1.55, z=1.05)),
        )
    baseline = RECORDS[sid][0]
    gate = HELDOUT[sid]['subset64_160_best_saved']['gate']
    fig.update_layout(
        title=(
            f'64/160/32 label-free future generation | SID {sid} | true diagnosis={baseline["diagnosis"]} | inferred p(disease)={gate:.3f}'
            '<br><sup>The prediction row uses only the first scan anchor and inferred disease probability.</sup>'
        ),
        width=260 * n,
        height=620,
        margin=dict(l=90, r=20, t=90, b=20),
    )
    return fig

if RUN_64_160_FUTURE_MESH_VIS:
    for sid in (healthy_sid, diseased_sid):
        save_plotly_figure(
            future_mesh_figure(sid),
            f'10_64_160_future_mesh_sid_{sid}',
            f'64/160/32 future meshes SID {sid}',
            'Future generation',
            'Top row is real GT scan. Bottom row is label-free prediction using inferred disease probability.',
        )
else:
    print('Skipped mesh future visualization. Set RUN_64_160_FUTURE_MESH_VIS=True.')


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/10_64_160_future_mesh_sid_27.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/10_64_160_future_mesh_sid_4.html


## Decoder Jacobian Sensitivity

This checks whether latent directions are amplified or suppressed by the decoder on real held-out surfaces.

In [10]:
# This cell computes decoder sensitivity and a simple Jacobian deformation proxy.
# It checks whether large latent velocities are actually amplified or suppressed by the INR decoder.
def latent_jacobian_norm(decoder, z, vertices, surface_batch=512):
    norms = []
    for start in range(0, vertices.shape[0], surface_batch):
        x = vertices[start:start + surface_batch].detach()
        zz = z.detach().expand(x.shape[0], -1).clone().requires_grad_(True)
        sdf = decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        # Decoder rows are independent, so grad of summed SDF gives each point's dG/dz row.
        grad_z = torch.autograd.grad(sdf.sum(), zz, create_graph=False, retain_graph=False)[0]
        norms.append(torch.linalg.norm(grad_z, dim=1).detach().cpu())
    return torch.cat(norms).numpy()

jac_rows = []
for sid in (healthy_sid, diseased_sid):
    for model_name, model in MODELS.items():
        item = HELDOUT[sid][model_name]
        norms = latent_jacobian_norm(model['decoder'], item['anchor'], item['geometry']['vertices'])
        jac_rows.append({
            'sid': sid,
            'model': model['label'],
            'mean_latent_jacobian_norm': float(np.mean(norms)),
            'p95_latent_jacobian_norm': float(np.quantile(norms, 0.95)),
            'max_latent_jacobian_norm': float(np.max(norms)),
        })
JACOBIAN_SENSITIVITY = pd.DataFrame(jac_rows)
save_table(
    JACOBIAN_SENSITIVITY,
    '11_jacobian_sensitivity_table',
    'Decoder latent-Jacobian sensitivity table',
    'Jacobian sensitivity',
    'Measures how strongly the INR changes with latent perturbations on held-out surfaces.',
)

fig, ax = plt.subplots(figsize=(10, 4))
for model, sub in JACOBIAN_SENSITIVITY.groupby('model'):
    ax.plot(sub['sid'].astype(str), sub['mean_latent_jacobian_norm'], marker='o', label=model)
ax.set_title('Mean decoder latent-Jacobian norm on held-out surfaces')
ax.set_xlabel('test subject')
ax.set_ylabel('||d INR / dz||')
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
save_matplotlib_figure(
    fig,
    '12_jacobian_sensitivity_plot',
    'Decoder latent-Jacobian sensitivity plot',
    'Jacobian sensitivity',
    'Large latent velocity is meaningful only if the decoder Jacobian maps it into visible shape change.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/11_jacobian_sensitivity_table.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/12_jacobian_sensitivity_plot.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/12_jacobian_sensitivity_plot.html')

## Part 2 Interpretation Guide

Use this notebook for subject-level conclusions:

- **Only `64/160/32` is label-free for diagnosis.** Old `64/128/64` and additive-best do not infer disease, so their held-out maps use oracle diagnosis. That is why the `gate_mode` column matters.
- **Real-vs-model speed:** observed thickness/bump rates show what actually changes between scans. Model RMS normal speed shows where each branch wants to move the surface.
- **Good disentanglement target:** healthy subjects should have low gated disease speed; diseased subjects should show disease speed without residual dominating.
- **Forecast target:** the 64/160/32 future table reports SDF MAE and volume error after using only the first scan and inferred disease probability.
- **Failure diagnosis:** if inferred disease probability is wrong but oracle maps look anatomically plausible, the disease classifier is the bottleneck. If both are poor, the flow/loss formulation is the bottleneck.


## Export Index
Run this after the cells above to refresh the indexed HTML report.


In [11]:
# Refresh the HTML index after running the output-saving cells above.
index_path = write_figure_index()
relative_index = index_path.relative_to(ROOT)
display(HTML(
    f"<p><strong>All Part 2 outputs:</strong> "
    f"<a href='{relative_index.as_posix()}' target='_blank'>{relative_index.as_posix()}</a></p>"
))
print('Saved Part 2 analysis:', OUTPUT_DIR)
print('Open this file for every Part 2 visualization/table:', index_path)


Figure index: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/index.html


Saved Part 2 analysis: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2
Open this file for every Part 2 visualization/table: /home/jakaria/INR/Deep3DComp/analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2/figures/index.html
